In [1]:
import pandas as pd

df = pd.read_parquet("merged_cleaned.parquet")

In [2]:
df = pd.read_parquet("merged_cleaned.parquet")

In [3]:
df

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_ece_packets,dst2src_urg_packets,dst2src_ack_packets,dst2src_psh_packets,dst2src_rst_packets,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence
0,0,0,192.168.137.85,a8:6b:ad:1f:9b:e5,a8:6b:ad,33682,192.168.137.198,dc:a6:32:c9:e5:5e,dc:a6:32,22,...,0,0,7,3,0,1,SSH,RemoteAccess,0,6
1,1,0,192.168.137.85,a8:6b:ad:1f:9b:e5,a8:6b:ad,47450,192.168.137.198,dc:a6:32:c9:e5:5e,dc:a6:32,22,...,0,0,1,0,0,0,SSH,RemoteAccess,1,1
2,2,0,fe80::813f:cb21:c6d4:b7c2,dc:a6:32:c9:e5:5e,dc:a6:32,49152,fe80::6c9b:9e0d:591c:fc8a,dc:a6:32:c9:e6:9f,dc:a6:32,34084,...,0,0,6429,6403,0,0,TLS,Web,0,6
3,3,0,192.168.137.198,dc:a6:32:c9:e5:5f,dc:a6:32,33234,192.168.137.87,dc:a6:32:c9:e5:3e,dc:a6:32,8765,...,0,0,348,326,0,0,Unknown,Unspecified,0,0
4,4,0,192.168.137.85,a8:6b:ad:1f:9b:e5,a8:6b:ad,50534,192.168.137.198,dc:a6:32:c9:e5:5f,dc:a6:32,139,...,0,0,1,0,1,0,NetBIOS,System,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2196811,57053,0,192.168.10.4,dc:a6:32:c9:e6:9f,dc:a6:32,56979,192.168.10.2,dc:a6:32:c9:e5:5e,dc:a6:32,7938,...,0,0,1,0,1,0,Unknown,Unspecified,0,0
2196812,57054,0,192.168.10.4,dc:a6:32:c9:e6:9f,dc:a6:32,56979,192.168.10.2,dc:a6:32:c9:e5:5e,dc:a6:32,6543,...,0,0,1,0,1,0,Unknown,Unspecified,0,0
2196813,57055,0,192.168.10.4,dc:a6:32:c9:e6:9f,dc:a6:32,56979,192.168.10.2,dc:a6:32:c9:e5:5e,dc:a6:32,8000,...,0,0,1,0,1,0,Unknown,Unspecified,0,0
2196814,57056,0,192.168.10.4,dc:a6:32:c9:e6:9f,dc:a6:32,56979,192.168.10.2,dc:a6:32:c9:e5:5e,dc:a6:32,5226,...,0,0,1,0,1,0,Unknown,Unspecified,0,0


In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.utils.multiclass import unique_labels

# Load cleaned dataset
df = pd.read_parquet("merged_cleaned.parquet")

# Set target column
target = 'application_name'

# Drop non-informative or identifier-like columns
drop_cols = ['id', 'src_ip', 'dst_ip', 'src_mac', 'dst_mac',
             'src_oui', 'dst_oui', 'application_category_name', 'application_is_guessed']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

# Encode target labels
label_encoder = LabelEncoder()
df[target] = label_encoder.fit_transform(df[target])

# Feature & target selection
X = df.drop(columns=[target]).select_dtypes(include=['number'])
y = df[target]

# Train/test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Decision Tree model (with depth/leaf constraints to prevent overfitting)
clf = DecisionTreeClassifier(
     max_depth=7,               # reduce depth further
    min_samples_leaf=10,        # prevent tiny leaf nodes
    min_samples_split=25,       # enforce a decent split
    class_weight='balanced', # helpful in imbalanced datasets
    random_state=42
)
clf.fit(X_train, y_train)

# Predict
y_pred = clf.predict(X_test)

# Evaluation
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2))
print("✅ Precision:", round(precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2))
print("✅ Recall:", round(recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2))
print("✅ F1 Score:", round(f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2))

# Classification report
labels_in_use = unique_labels(y_test, y_pred)
class_names_in_use = label_encoder.inverse_transform(labels_in_use)

print("\n📄 Classification Report:\n", classification_report(
    y_test, y_pred,
    labels=labels_in_use,
    target_names=class_names_in_use,
    zero_division=0
))


✅ Accuracy: 95.64
✅ Precision: 95.53
✅ Recall: 95.64
✅ F1 Score: 95.54

📄 Classification Report:
                  precision    recall  f1-score   support

            AFP       0.00      0.00      0.00       330
            AJP       0.00      0.00      0.00       660
            BGP       0.00      0.00      0.00       661
    CiscoSkinny       0.00      0.00      0.00       331
       CiscoVPN       0.00      0.00      0.00       660
         Citrix       0.00      0.00      0.00       331
           DHCP       1.00      1.00      1.00         4
           DNP3       0.00      0.00      0.00       329
            DNS       0.00      0.00      0.00       329
  DNS.UbuntuONE       1.00      0.88      0.94        17
    FTP_CONTROL       0.00      0.00      0.00       329
       FTP_DATA       0.50      1.00      0.67       329
            Git       0.02      1.00      0.03       331
           H323       0.00      0.00      0.00       660
           HTTP       0.00      0.00      0.00